In [1]:
import pandas as pd
import numpy as np

import joblib

from sklearn.model_selection import train_test_split

from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score)

In [2]:
df = pd.read_csv(r"E:\AARAV\Infotact-DS-ML\Project-1-Predictive-Maintenance\data\processed\model_ready_dataset.csv")

df.head()

,HDF,OSF,PWF,TWF,rpm_torque_interaction,Rotational speed [rpm],load_stress,Torque [Nm],load_density,Tool wear [min],temperature_ratio,tool_wear_mean_10,temperature_difference,air_temp_mean_10,UDI,Machine failure
0,0,0,0,0,71177.0,1306,29.7025,54.5,0.545,50,1.034806,36.8,10.4,298.60,19,0
1,0,0,0,0,53040.0,1632,10.5625,32.5,0.325,55,1.034794,40.2,10.4,298.64,20,0
2,0,0,0,0,58712.5,1375,18.2329,42.7,0.427,58,1.034794,43.6,10.4,298.69,21,0
3,0,0,0,0,64960.0,1450,20.0704,44.8,0.448,63,1.035141,47.0,10.5,298.71,22,0
4,0,0,0,0,48536.7,1581,9.4249,30.7,0.307,65,1.034794,50.1,10.4,298.74,23,0


In [3]:
X = df.drop("Machine failure", axis=1)

y = df["Machine failure"]

In [4]:
X.columns = (
    X.columns
    .str.replace("[","",regex=False)
    .str.replace("]","",regex=False)
    .str.replace("{","",regex=False)
    .str.replace("}","",regex=False)
    .str.replace(":","",regex=False)
    .str.replace(",","",regex=False)
)

In [5]:
leak_columns = [
    "Machine failure",
    "TWF",
    "HDF",
    "PWF",
    "OSF",
    "RNF"
]

X = X.drop(
    columns=[
        col for col in leak_columns
        if col in X.columns
    ]
)

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X,y,train_size=0.2,random_state=42,stratify=y)

In [7]:
model = joblib.load(r"E:\AARAV\Infotact-DS-ML\Project-1-Predictive-Maintenance\models\final_lightgbm_model.pkl")

In [8]:
def add_noise(data, level):
    noise = np.random.normal(0,level,data.shape)
    return data + noise

In [9]:
best_threshold = 0.8000000

clean_prob = model.predict_proba(X_test)[:,1]

clean_pred = (clean_prob>=best_threshold).astype(int)

clean_result = {
    "Dataset":"Clean",
    "Accuracy":accuracy_score(y_test,clean_pred),
    "Precision":precision_score(y_test,clean_pred),
    "Recall":recall_score(y_test,clean_pred),
    "F1":f1_score(y_test,clean_pred)
}

In [10]:
noisy_test = add_noise(X_test,0.05)

noisy_prob = model.predict_proba(noisy_test)[:,1]

noisy_pred = (noisy_prob>=best_threshold).astype(int)

noisy_result = {
    "Dataset":"Noisy",
    "Accuracy":accuracy_score(y_test,noisy_pred),
    "Precision":precision_score(y_test,noisy_pred),
    "Recall":recall_score(y_test,noisy_pred),
    "F1":f1_score(y_test,noisy_pred)
}

In [11]:
comparison = pd.DataFrame([clean_result,noisy_result])

comparison

,Dataset,Accuracy,Precision,Recall,F1
0,Clean,0.995743,0.961089,0.911439,0.935606
1,Noisy,0.986852,0.753049,0.911439,0.824708


In [12]:
comparison.to_csv(r"E:\AARAV\Infotact-DS-ML\Project-1-Predictive-Maintenance\data\processed\robustness_comparison.csv",index=False)